## 模块 5：客户价值分层与消费画像分析（分桶分层）

### 业务需求

基于原始表，按客户累计消费金额分为高 / 中 / 低价值三层，统计各层级客户画像指标

### 思路分析

1. 直接关联销售订单表、客户信息表，按客户 ID 分组统计消费指标

2. 用`NTILE(3)`窗口函数按消费金额分 3 个层级，实现客户价值分层

3. 用`CASE WHEN`给层级打标签，统计各层级整体画像

### 核心知识点

- 窗口函数`NTILE()`分桶分层

- 聚合函数嵌套使用

- `CASE WHEN`条件打标签

- 用户分层分析逻辑

In [1]:
import pandas as pd
import sqlite3

# 1. 连接SQLite数据库
conn = sqlite3.connect("/kaggle/input/datasets/tclaim/retail-sales/retail_sales.db")

# 4. 封装通用SQL执行函数，自动打印表格结果
def query_sql(sql):
    return pd.read_sql(sql, conn)
    
print("数据导入完成，调用 query_sql('SQL语句') 即可执行查询")

数据导入完成，调用 query_sql('SQL语句') 即可执行查询


In [2]:
sql = '''
WITH customer_consumption AS (
    SELECT
        o.客户ID,
        c.会员等级,
        SUM(o.销售数量 * o."单价(元)") AS 累计消费金额,
        COUNT(o.订单ID) AS 订单频次,
        AVG(o.销售数量 * o."单价(元)") AS 平均客单价
    FROM "销售订单表" o
    INNER JOIN "客户信息表" c ON o.客户ID = c.客户ID
    GROUP BY o.客户ID, c.会员等级
),
customer_tier AS (
    SELECT
        *,
        NTILE(3) OVER (ORDER BY 累计消费金额 DESC) AS 价值层级
    FROM customer_consumption
)
SELECT
    CASE 价值层级
        WHEN 1 THEN '高价值客户'
        WHEN 2 THEN '中价值客户'
        WHEN 3 THEN '低价值客户'
    END AS 客户价值层级,
    COUNT(客户ID) AS 客户数,
    SUM(累计消费金额) AS 层级总消费,
    AVG(平均客单价) AS 层级平均客单价,
    AVG(订单频次) AS 层级平均订单频次
FROM customer_tier
GROUP BY 价值层级
ORDER BY 价值层级;
'''
query_sql(sql)

,客户价值层级,客户数,层级总消费,层级平均客单价,层级平均订单频次
0,高价值客户,103,12283188.0,77046.669903,1.902913
1,中价值客户,103,828515.0,5568.734628,1.699029
2,低价值客户,103,96672.5,765.881877,1.252427


### 结果说明

输出客户价值分层结果，清晰展示高 / 中 / 低价值客户的数量、总消费、客单价、订单频次，可直接用于用户运营策略制定，针对不同层级客户制定差异化运营方案